# Analyse du biais product_type

Ce notebook vérifie si les features internes du CNN contiennent beaucoup d'information sur le `product_type`.

L'idée est simple : on garde le CNN entraîné pour `fresh` / `rotten`, puis on utilise ses features pour entraîner un petit classifieur qui prédit le type du produit.

Si ce classifieur fonctionne bien, cela suggère que le CNN n'apprend pas seulement la fraîcheur. Il encode aussi fortement les catégories de produits.

## Principe du linear probe

Le CNN n'est pas réentraîné.

```text
image -> CNN gelé -> features internes -> LogisticRegression -> product_type
```

Cette méthode sert à analyser ce que les représentations du modèle contiennent déjà.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

reports_dir = project_root / "reports"
project_root

## Lancer l'analyse

Cette cellule utilise le modèle `baseline_model.keras` et le `standard_split` déjà sauvegardés.

In [ ]:
command = [sys.executable, str(project_root / "src" / "feature_product_type_analysis.py")]
subprocess.run(command, cwd=project_root, check=True)

## Résultats globaux

On compare la précision du probe avec une baseline très simple : prédire toujours le `product_type` majoritaire du `training_set`.

In [ ]:
probe_metrics = pd.read_csv(reports_dir / "product_type_probe_metrics.csv")
probe_metrics

In [ ]:
plot_columns = ["majority_baseline_accuracy", "accuracy", "balanced_accuracy", "macro_f1_score"]
ax = probe_metrics.set_index("split")[plot_columns].plot(kind="bar", figsize=(9, 5))
ax.set_ylim(0, 1)
ax.set_title("Prédiction du product_type depuis les features du CNN")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Résultats par product_type

In [ ]:
product_type_metrics = pd.read_csv(reports_dir / "product_type_probe_by_product_type.csv")
product_type_metrics.sort_values("f1_score")

In [ ]:
ax = product_type_metrics.sort_values("f1_score").plot(
    x="product_type",
    y="f1_score",
    kind="barh",
    figsize=(8, 8),
    legend=False,
)
ax.set_xlim(0, 1)
ax.set_title("F1-score du probe par product_type")
ax.set_xlabel("F1-score")
ax.set_ylabel("product_type")
plt.tight_layout()
plt.show()

## Matrice de confusion

La matrice de confusion permet de voir quels types de produits sont confondus dans l'espace de features.

In [ ]:
confusion_df = pd.read_csv(reports_dir / "product_type_probe_confusion_matrix.csv", index_col=0)
confusion_df

## Conclusion à compléter

Si le probe prédit bien `product_type`, cela soutient l'hypothèse suivante :

Le CNN encode fortement le type de produit dans ses représentations internes. Le modèle peut donc obtenir de bons résultats sur le `standard_split` en utilisant aussi des indices liés aux catégories vues pendant l'entraînement.

Cette analyse aide à expliquer pourquoi la performance baisse sur le `unseen_category_split`.